# Terrain & geology — where to look for landslides

Adds two targeting layers to the imagery workflow:

- **Slope** from the [Copernicus GLO-30 DEM](https://registry.opendata.aws/copernicus-dem/) (30 m, © DLR/ESA) — steep terrain is where earthquake-triggered landslides concentrate. Only the AOI is downloaded (windowed reads from cloud COGs), slope is computed locally in UTM 19N.
- **Surface geology** from [Macrostrat](https://macrostrat.org) (CC-BY), which in Venezuela serves the digitized **USGS Geologic Map of Venezuela, 1:750,000** (Hackley, Urbani, Karlsen & Garrity 2005 — [OFR 2005-1038](https://pubs.usgs.gov/of/2005/1038/) / [DS-199](https://pubs.usgs.gov/publication/ds199)). Weak schists of the coastal ranges (e.g. Tacagua Schist) fed the 1999 Vargas debris flows — lithology matters here.

Run top to bottom; change `LOCATION` to switch areas. DEM/slope rasters are cached in `data/terrain/`.

In [1]:
from pathlib import Path

import leafmap

from geer_venezuela import (
    ATTRIBUTION,
    GEOLOGY_ATTRIBUTION,
    MACROSTRAT_TILE_URL,
    asset_href,
    compute_slope,
    fetch_dem,
    fetch_geology,
    geology_at,
    load_items,
    scenes_for,
    steep_areas,
)

DATA = (Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent) / "data"

LOCATION = "la-guaira"
BUFFER_DEG = 0.02  # ~2 km context around the imagery footprints
STEEP_DEG = 30  # slope threshold for "steep" outlines

items = load_items("post-event")
scenes = scenes_for(items, LOCATION)
minx, miny, maxx, maxy = scenes.total_bounds
bounds = (minx - BUFFER_DEG, miny - BUFFER_DEG, maxx + BUFFER_DEG, maxy + BUFFER_DEG)
print(f"{LOCATION}: {len(scenes)} scenes, AOI bounds {[round(b, 3) for b in bounds]}")

la-guaira: 8 scenes, AOI bounds [np.float64(-67.119), np.float64(10.401), np.float64(-66.722), np.float64(10.746)]


## 1. DEM → slope

First run per location downloads the DEM clip (a few MB) and computes slope; afterwards it's cached.

In [2]:
import rasterio

dem_file = fetch_dem(bounds, DATA / "terrain" / f"{LOCATION}_dem.tif")
slope_file = compute_slope(dem_file, DATA / "terrain" / f"{LOCATION}_slope.tif")

with rasterio.open(slope_file) as src:
    slope_data = src.read(1)
print(
    f"slope: mean {slope_data.mean():.1f}°, "
    f"{(slope_data >= STEEP_DEG).mean() * 100:.1f}% of AOI ≥ {STEEP_DEG}°"
)

steep = steep_areas(slope_file, threshold=STEEP_DEG)
print(f"{len(steep)} steep polygons, {steep['area_km2'].sum():.0f} km² total")

slope: mean 13.0°, 14.1% of AOI ≥ 30°
1709 steep polygons, 219 km² total


## 2. Surface geology in the AOI

Units present, largest first. Schists and phyllites on steep slopes are the classic failure combination on this coast.

In [ ]:
geology = fetch_geology(bounds)

summary = (
    geology.assign(area_km2=geology.to_crs(32619).area / 1e6)
    .groupby(["name", "lith", "age"], as_index=False)["area_km2"]
    .sum()
    .sort_values("area_km2", ascending=False)
    .round(1)
)
summary

## 3. Combined targeting map

Layers (toggle in the control, top right):

- **AFTER imagery** — post-event 50 cm scene
- **Slope** — 0–45°, darker = flatter, brighter = steeper
- **Steep areas** — red outlines where slope ≥ threshold: search here first
- **Geology** — unit polygons (hover for name/lithology); plus Macrostrat's own rendered tiles as an alternate view

Use the draw tools to mark candidate areas, then run the export cell.

In [ ]:
m = leafmap.Map()
m.add_cog_layer(
    asset_href(scenes.iloc[0], "visual"),
    name=f"AFTER — {scenes.iloc[0]['title']}",
    attribution=ATTRIBUTION,
)
m.add_raster(
    str(slope_file),
    colormap="inferno",
    vmin=0,
    vmax=45,
    opacity=0.6,
    layer_name="Slope (degrees)",
    zoom_to_layer=False,
)
m.add_gdf(
    steep,
    layer_name=f"Steep areas (≥{STEEP_DEG}°)",
    style={"color": "#ff3b30", "weight": 1.5, "fillOpacity": 0.25},
    info_mode=None,
    zoom_to_layer=False,
)
m.add_gdf(
    geology,
    layer_name="Geology (USGS 1:750k via Macrostrat)",
    style_callback=lambda feat: {
        "fillColor": feat["properties"]["color"] or "#cccccc",
        "color": "#555555",
        "weight": 0.5,
        "fillOpacity": 0.35,
    },
    info_mode="on_hover",
    zoom_to_layer=False,
)
m.add_tile_layer(
    MACROSTRAT_TILE_URL,
    name="Macrostrat rendered geology",
    attribution=GEOLOGY_ATTRIBUTION,
    shown=False,
    opacity=0.7,
)
m

Map(center=[-66.98433806397188, 10.540821876664467], controls=(ZoomControl(options=['position', 'zoom_in_text'…

## 4. Export for field teams / QGIS

Steep-area outlines and geology go to `data/` as GeoJSON; anything you drew on the map above is saved alongside the landslide candidates from notebook 01.

In [ ]:
steep_file = DATA / "terrain" / f"{LOCATION}_steep{STEEP_DEG}.geojson"
steep.to_file(steep_file)
geology_file = DATA / "terrain" / f"{LOCATION}_geology.geojson"
geology.to_file(geology_file)
print(f"Saved {steep_file.name} and {geology_file.name} in {steep_file.parent}")

if m.user_rois is not None and m.user_rois["features"]:
    drawn_file = DATA / "landslide_candidates" / f"{LOCATION}_terrain_targets.geojson"
    m.save_draw_features(str(drawn_file), indent=2)
    print(f"Saved {len(m.user_rois['features'])} drawn feature(s) to {drawn_file}")

Saved la-guaira_steep30.geojson and la-guaira_geology.geojson in /Users/lornearnold/GitHub/GEER_Venezuela/data/terrain


## 5. What's under a specific point?

Quick lookup for any coordinate (e.g. a suspected landslide) — returns the geologic unit(s) from Macrostrat.

In [ ]:
geology_at(10.60, -66.93)  # example: slopes above La Guaira

,name,strat_name,lith,t_int_name,b_int_name
0,"Tacagua Schist, Antímano Marble, undivided of ...",Tacagua Schist; Antímano Marble,schist and marble,Cretaceous,Cretaceous
1,Mesozoic crystalline metamorphic rocks,,medium-high grade orthogneiss,Mesozoic,Mesozoic
2,Mesozoic sedimentary,,sedimentary,Mesozoic,Mesozoic


## Caveats

- GLO-30 is a **surface** model (DSM): slopes include canopy and buildings, and 30 m smooths out small scarps — treat the slope layer as a *search prioritizer*, not a precise stability map.
- The geology map is 1:750,000 — unit boundaries can be off by hundreds of meters at these zoom levels.
- Steep-area polygons under 0.01 km² are dropped by default (`steep_areas(min_area_m2=...)` to change).

**Sources**: Copernicus GLO-30 DEM © DLR/ESA (AWS Open Data); geology © Macrostrat (CC-BY) serving USGS OFR 2005-1038 / DS-199 (Hackley, Urbani, Karlsen & Garrity).